In [7]:
# ==========================================================
# 05 - Surrogate Utility Classifier
# Imports and basic configuration
# ==========================================================

import os
import sys

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

# Reproducibility
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

# Use GPU when available; otherwise use CPU
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch version:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("CUDA is not available; using CPU.")

PyTorch version: 2.12.1+cpu
Device: cpu
CUDA is not available; using CPU.


In [8]:
from pathlib import Path
import os

PROCESSED_DIR = Path(
    os.environ.get(
        "IMU_PROCESSED_PATH",
        "../datasets/DatasetIMUandBIOMARKERS/processed"
    )
).expanduser().resolve()

NUM_CLASSES = 6
Z_DIM = 60

BATCH_SIZE = 8 if device.type == "cpu" else 64

print("Number of Classes :", NUM_CLASSES)
print("Latent Dimension  :", Z_DIM)
print("Batch Size        :", BATCH_SIZE)
print("Processed Folder  :", PROCESSED_DIR)

if not PROCESSED_DIR.exists():
    raise FileNotFoundError(
        f"Processed dataset directory not found:\n{PROCESSED_DIR}"
    )

Number of Classes : 6
Latent Dimension  : 60
Batch Size        : 8
Processed Folder  : C:\PrivDiffuser_Narval\datasets\DatasetIMUandBIOMARKERS\processed


In [9]:
# ==========================================================
# Configuration: Small CPU or Full GPU/Narval
# ==========================================================

from pathlib import Path
import os
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Automatically choose mode from the available device.
# CPU  -> small balanced one-subject experiment
# CUDA -> full multi-subject experiment
RUN_MODE = "full_gpu" if device.type == "cuda" else "small_cpu"

PROCESSED_DIR = Path(
    os.environ.get(
        "IMU_PROCESSED_PATH",
        "../datasets/DatasetIMUandBIOMARKERS/processed"
    )
).expanduser().resolve()

NUM_ACTIVITIES = 6
NUM_GENDERS = 2
NUM_WEIGHT_CLASSES = 3
Z_DIM = 60

if RUN_MODE == "small_cpu":
    BATCH_SIZE = 8

    TRAIN_FILE = (
        PROCESSED_DIR
        / "Subject02_train.npz"
    )

    TEST_FILE = (
        PROCESSED_DIR
        / "Subject01_test.npz"
    )

    TRAIN_FILES = [TRAIN_FILE]
    TEST_FILES = [TEST_FILE]

    TRAIN_SAMPLES_PER_CLASS = 100
    TEST_SAMPLES_PER_CLASS = 50

else:
    BATCH_SIZE = 64

    TRAIN_FILES = sorted(
        PROCESSED_DIR.glob("*_train.npz")
    )

    TEST_FILES = sorted(
        PROCESSED_DIR.glob("*_test.npz")
    )

    TRAIN_SAMPLES_PER_CLASS = None
    TEST_SAMPLES_PER_CLASS = None


print("=" * 70)
print("SURROGATE MODEL CONFIGURATION")
print("=" * 70)

print("Device               :", device)
print("Run mode             :", RUN_MODE)
print("Processed directory  :", PROCESSED_DIR)
print("Batch size           :", BATCH_SIZE)
print("Training files       :", len(TRAIN_FILES))
print("Testing files        :", len(TEST_FILES))

if RUN_MODE == "small_cpu":
    print("Training participant :", TRAIN_FILES[0].name)
    print("Testing participant  :", TEST_FILES[0].name)
    print(
        "Train samples/class :",
        TRAIN_SAMPLES_PER_CLASS
    )
    print(
        "Test samples/class  :",
        TEST_SAMPLES_PER_CLASS
    )
else:
    print("Using all processed training and test subjects.")

if not PROCESSED_DIR.exists():
    raise FileNotFoundError(
        f"Processed dataset folder not found:\n"
        f"{PROCESSED_DIR}"
    )

for file_path in TRAIN_FILES + TEST_FILES:
    if not file_path.exists():
        raise FileNotFoundError(
            f"Required file not found: {file_path}"
        )

if RUN_MODE == "full_gpu":
    assert len(TRAIN_FILES) == 48, (
        f"Expected 48 training files, "
        f"found {len(TRAIN_FILES)}"
    )

    assert len(TEST_FILES) == 12, (
        f"Expected 12 testing files, "
        f"found {len(TEST_FILES)}"
    )

SURROGATE MODEL CONFIGURATION
Device               : cpu
Run mode             : small_cpu
Processed directory  : C:\PrivDiffuser_Narval\datasets\DatasetIMUandBIOMARKERS\processed
Batch size           : 8
Training files       : 1
Testing files        : 1
Training participant : Subject02_train.npz
Testing participant  : Subject01_test.npz
Train samples/class : 100
Test samples/class  : 50


In [19]:
# ==========================================================
# Cell 4 - Surrogate Utility Classifier
# ==========================================================

import torch
import torch.nn as nn
import torch.nn.functional as F


class SmallSurrogateClassifier(nn.Module):
    """
    Lightweight CNN for six-class activity recognition.

    Input shape:
        (batch_size, 1, 128, 30)

    Outputs:
        activity_logits:
            Shape (batch_size, num_classes)

        z_public:
            Public/utility embedding with shape
            (batch_size, z_dim)
    """

    def __init__(
        self,
        num_classes=6,
        z_dim=60
    ):
        super().__init__()

        # --------------------------------------------------
        # CNN feature extractor
        # --------------------------------------------------
        self.feature_extractor = nn.Sequential(

            # Input:
            # (batch, 1, 128, 30)
            nn.Conv2d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),

            # Output after pooling:
            # (batch, 16, 64, 15)
            nn.MaxPool2d(
                kernel_size=2
            ),

            nn.Conv2d(
                in_channels=16,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),

            # Convert every feature map into one value
            # Output:
            # (batch, 32, 1, 1)
            nn.AdaptiveAvgPool2d(
                output_size=(1, 1)
            )
        )

        # --------------------------------------------------
        # Public utility embedding
        # --------------------------------------------------
        self.embedding_layer = nn.Linear(
            in_features=32,
            out_features=z_dim
        )

        # --------------------------------------------------
        # Six-class activity classifier
        # --------------------------------------------------
        self.activity_classifier = nn.Linear(
            in_features=z_dim,
            out_features=num_classes
        )

    def forward(self, x):

        # Extract CNN features
        features = self.feature_extractor(x)

        # Convert:
        # (batch, 32, 1, 1)
        # into:
        # (batch, 32)
        features = torch.flatten(
            features,
            start_dim=1
        )

        # Create 60-dimensional public representation
        z_public = F.relu(
            self.embedding_layer(features)
        )

        # Predict one of six activities
        activity_logits = self.activity_classifier(
            z_public
        )

        return activity_logits, z_public


# ==========================================================
# Create Model
# ==========================================================

model = SmallSurrogateClassifier(
    num_classes=NUM_CLASSES,
    z_dim=Z_DIM
).to(device)


print("=" * 70)
print("SURROGATE UTILITY CLASSIFIER")
print("=" * 70)

print("Run mode :", RUN_MODE)
print("Device   :", device)
print("Classes  :", NUM_CLASSES)
print("Z dimension:", Z_DIM)

print("\nModel architecture:")
print(model)


# ==========================================================
# Verify Model Using One Batch
# ==========================================================

sample_windows, _, _, _ = next(
    iter(surrogate_train_loader)
)

model.eval()

with torch.no_grad():

    sample_windows_device = sample_windows.to(
        device,
        dtype=torch.float32,
        non_blocking=True
    )

    test_logits, test_embedding = model(
        sample_windows_device
    )

assert test_logits.shape == (
    sample_windows.shape[0],
    NUM_CLASSES
)

assert test_embedding.shape == (
    sample_windows.shape[0],
    Z_DIM
)

print("\n" + "=" * 70)
print("MODEL OUTPUT VERIFICATION")
print("=" * 70)

print(
    "Input window shape     :",
    tuple(sample_windows.shape)
)

print(
    "Activity logits shape  :",
    tuple(test_logits.shape)
)

print(
    "Public embedding shape :",
    tuple(test_embedding.shape)
)

print("\nSurrogate classifier verified successfully.")

SURROGATE UTILITY CLASSIFIER
Run mode : small_cpu
Device   : cpu
Classes  : 6
Z dimension: 60

Model architecture:
SmallSurrogateClassifier(
  (feature_extractor): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): AdaptiveAvgPool2d(output_size=(1, 1))
  )
  (embedding_layer): Linear(in_features=32, out_features=60, bias=True)
  (activity_classifier): Linear(in_features=60, out_features=6, bias=True)
)

MODEL OUTPUT VERIFICATION
Input window shape     : (8, 1, 128, 30)
Activity logits shape  : (8, 6)
Public embedding shape : (8, 60)

Surrogate classifier verified successfully.


In [20]:
# ==========================================================
# Cell 5 - Train Surrogate Utility Classifier
# ==========================================================

import time
from pathlib import Path

# ----------------------------------------------------------
# Training configuration
# ----------------------------------------------------------

criterion = nn.CrossEntropyLoss()

LEARNING_RATE = 0.001

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

# Keep the existing quick CPU experiment.
# Use more epochs when the full GPU mode is selected.
if RUN_MODE == "small_cpu":
    EPOCHS = 5
    train_loader = quick_train_loader
    model_filename = "small_surrogate_activity_model.pt"
else:
    EPOCHS = 30
    train_loader = full_train_loader
    model_filename = "full_surrogate_activity_model.pt"

MODEL_SAVE_PATH = Path(model_filename)


print("=" * 75)
print("SURROGATE UTILITY CLASSIFIER TRAINING")
print("=" * 75)

print("Run mode      :", RUN_MODE)
print("Device        :", device)
print("Epochs        :", EPOCHS)
print("Learning rate :", LEARNING_RATE)
print("Batch size    :", BATCH_SIZE)
print("Model file    :", MODEL_SAVE_PATH)


# ----------------------------------------------------------
# Store training history
# ----------------------------------------------------------

training_losses = []
training_accuracies = []

start_time = time.time()


# ----------------------------------------------------------
# Training loop
# ----------------------------------------------------------

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for windows, activity, gender, weight in train_loader:

        windows = windows.to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        # Activity labels were stored as one-hot vectors.
        # Convert them into integer class IDs: 0–5.
        activity_targets = torch.argmax(
            activity,
            dim=1
        ).to(
            device,
            dtype=torch.long,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        activity_logits, z_public = model(
            windows
        )

        loss = criterion(
            activity_logits,
            activity_targets
        )

        loss.backward()
        optimizer.step()

        batch_size = activity_targets.size(0)

        running_loss += (
            loss.item() * batch_size
        )

        predicted_classes = torch.argmax(
            activity_logits,
            dim=1
        )

        correct_predictions += (
            predicted_classes
            == activity_targets
        ).sum().item()

        total_samples += batch_size


    epoch_loss = (
        running_loss / total_samples
    )

    epoch_accuracy = (
        correct_predictions / total_samples
    )

    training_losses.append(
        epoch_loss
    )

    training_accuracies.append(
        epoch_accuracy
    )

    print(
        f"Epoch {epoch + 1:02d}/{EPOCHS:02d} | "
        f"Loss: {epoch_loss:.4f} | "
        f"Training Accuracy: {epoch_accuracy:.4f}"
    )


# ----------------------------------------------------------
# Training summary
# ----------------------------------------------------------

training_time = (
    time.time() - start_time
)

print("\n" + "=" * 75)
print("TRAINING COMPLETED")
print("=" * 75)

print(
    f"Final training loss     : "
    f"{training_losses[-1]:.4f}"
)

print(
    f"Final training accuracy : "
    f"{training_accuracies[-1]:.4f}"
)

print(
    f"Total training time     : "
    f"{training_time:.2f} seconds"
)


# ----------------------------------------------------------
# Save trained model and configuration
# ----------------------------------------------------------

checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "training_losses": training_losses,
    "training_accuracies": training_accuracies,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "num_classes": NUM_CLASSES,
    "z_dim": Z_DIM,
    "run_mode": RUN_MODE
}

torch.save(
    checkpoint,
    MODEL_SAVE_PATH
)

print(
    "\nModel checkpoint saved as:",
    MODEL_SAVE_PATH
)

SURROGATE UTILITY CLASSIFIER TRAINING
Run mode      : small_cpu
Device        : cpu
Epochs        : 5
Learning rate : 0.001
Batch size    : 8
Model file    : small_surrogate_activity_model.pt
Epoch 01/05 | Loss: 1.7599 | Training Accuracy: 0.2017
Epoch 02/05 | Loss: 1.4710 | Training Accuracy: 0.4667
Epoch 03/05 | Loss: 1.0439 | Training Accuracy: 0.6333
Epoch 04/05 | Loss: 0.8639 | Training Accuracy: 0.6783
Epoch 05/05 | Loss: 0.8180 | Training Accuracy: 0.7150

TRAINING COMPLETED
Final training loss     : 0.8180
Final training accuracy : 0.7150
Total training time     : 1.57 seconds

Model checkpoint saved as: small_surrogate_activity_model.pt


In [21]:
# ==========================================================
# Cell 6 - Evaluate Surrogate Utility Classifier
# ==========================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# ----------------------------------------------------------
# Select the correct test loader
# ----------------------------------------------------------

if RUN_MODE == "small_cpu":
    test_loader = quick_test_loader
    training_sample_count = len(quick_train_dataset)
    testing_sample_count = len(quick_test_dataset)
else:
    test_loader = full_test_loader
    training_sample_count = len(full_train_dataset)
    testing_sample_count = len(full_test_dataset)


# ----------------------------------------------------------
# Run model evaluation
# ----------------------------------------------------------

model.eval()

true_labels = []
predicted_labels = []

with torch.no_grad():

    for windows, activity, gender, weight in test_loader:

        windows = windows.to(
            device,
            dtype=torch.float32,
            non_blocking=True
        )

        # Convert one-hot activity labels into class IDs 0–5
        activity_targets = torch.argmax(
            activity,
            dim=1
        ).to(
            device,
            dtype=torch.long,
            non_blocking=True
        )

        activity_logits, z_public = model(
            windows
        )

        predicted_classes = torch.argmax(
            activity_logits,
            dim=1
        )

        true_labels.extend(
            activity_targets.cpu().numpy()
        )

        predicted_labels.extend(
            predicted_classes.cpu().numpy()
        )


true_labels = np.asarray(true_labels)
predicted_labels = np.asarray(predicted_labels)


# ----------------------------------------------------------
# Calculate overall metrics
# ----------------------------------------------------------

accuracy = accuracy_score(
    true_labels,
    predicted_labels
)

macro_precision = precision_score(
    true_labels,
    predicted_labels,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    true_labels,
    predicted_labels,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    true_labels,
    predicted_labels,
    average="macro",
    zero_division=0
)

weighted_f1 = f1_score(
    true_labels,
    predicted_labels,
    average="weighted",
    zero_division=0
)


print("=" * 75)
print("SURROGATE ACTIVITY CLASSIFICATION RESULTS")
print("=" * 75)

print("Run mode            :", RUN_MODE)
print("Device              :", device)
print("Training samples    :", training_sample_count)
print("Testing samples     :", testing_sample_count)
print(f"Test Accuracy       : {accuracy:.4f}")
print(f"Macro Precision     : {macro_precision:.4f}")
print(f"Macro Recall        : {macro_recall:.4f}")
print(f"Macro F1-score      : {macro_f1:.4f}")
print(f"Weighted F1-score   : {weighted_f1:.4f}")


# ----------------------------------------------------------
# Per-activity classification report
# ----------------------------------------------------------

activity_names = [
    "Activity 0",
    "Activity 1",
    "Activity 2",
    "Activity 3",
    "Activity 4",
    "Activity 5"
]

print("\n" + "=" * 75)
print("PER-ACTIVITY CLASSIFICATION REPORT")
print("=" * 75)

print(
    classification_report(
        true_labels,
        predicted_labels,
        labels=[0, 1, 2, 3, 4, 5],
        target_names=activity_names,
        digits=4,
        zero_division=0
    )
)


# ----------------------------------------------------------
# Plain confusion matrix
# ----------------------------------------------------------

cm = confusion_matrix(
    true_labels,
    predicted_labels,
    labels=[0, 1, 2, 3, 4, 5]
)

confusion_matrix_table = pd.DataFrame(
    cm,
    index=[
        "True A0",
        "True A1",
        "True A2",
        "True A3",
        "True A4",
        "True A5"
    ],
    columns=[
        "Predicted A0",
        "Predicted A1",
        "Predicted A2",
        "Predicted A3",
        "Predicted A4",
        "Predicted A5"
    ]
)

print("=" * 90)
print("ACTIVITY CONFUSION MATRIX")
print("=" * 90)

print(
    confusion_matrix_table.to_string()
)


# ----------------------------------------------------------
# Optional CSV outputs for the report/GitHub
# ----------------------------------------------------------

results_summary = pd.DataFrame(
    [
        {
            "Run Mode": RUN_MODE,
            "Training Samples": training_sample_count,
            "Testing Samples": testing_sample_count,
            "Accuracy": accuracy,
            "Macro Precision": macro_precision,
            "Macro Recall": macro_recall,
            "Macro F1": macro_f1,
            "Weighted F1": weighted_f1
        }
    ]
)

results_summary.to_csv(
    "surrogate_activity_results.csv",
    index=False
)

confusion_matrix_table.to_csv(
    "surrogate_activity_confusion_matrix.csv"
)

print("\nSaved as:")
print("surrogate_activity_results.csv")
print("surrogate_activity_confusion_matrix.csv")

SURROGATE ACTIVITY CLASSIFICATION RESULTS
Run mode            : small_cpu
Device              : cpu
Training samples    : 600
Testing samples     : 300
Test Accuracy       : 0.5967
Macro Precision     : 0.6183
Macro Recall        : 0.5967
Macro F1-score      : 0.5426
Weighted F1-score   : 0.5426

PER-ACTIVITY CLASSIFICATION REPORT
              precision    recall  f1-score   support

  Activity 0     0.5000    0.1400    0.2188        50
  Activity 1     0.5333    0.9600    0.6857        50
  Activity 2     0.6486    0.9600    0.7742        50
  Activity 3     0.5217    0.7200    0.6050        50
  Activity 4     0.7500    0.1800    0.2903        50
  Activity 5     0.7561    0.6200    0.6813        50

    accuracy                         0.5967       300
   macro avg     0.6183    0.5967    0.5426       300
weighted avg     0.6183    0.5967    0.5426       300

ACTIVITY CONFUSION MATRIX
         Predicted A0  Predicted A1  Predicted A2  Predicted A3  Predicted A4  Predicted A5
True A

In [33]:
# ==========================================================
# IMU Dataset - Memory-Efficient Multi-Subject Loader
# ==========================================================

from pathlib import Path
import bisect

import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader


class IMUDataset(Dataset):
    """
    Memory-efficient dataset for processed subject-level
    IMU .npz files.

    The dataset does not combine all windows into RAM.
    It loads only the required subject file when a sample
    is requested.

    Each item returns:
        window:
            float32 tensor with shape (1, 128, 30)

        activity:
            integer class label from 0 to 5

        gender:
            integer class label from 0 to 1

        weight:
            integer class label from 0 to 2
    """

    def __init__(
        self,
        processed_dir,
        split="train"
    ):
        super().__init__()

        self.processed_dir = Path(
            processed_dir
        ).expanduser().resolve()

        self.split = split

        # --------------------------------------------------
        # Validate split and directory
        # --------------------------------------------------

        if self.split not in {
            "train",
            "test"
        }:
            raise ValueError(
                "split must be either "
                "'train' or 'test'."
            )

        if not self.processed_dir.exists():
            raise FileNotFoundError(
                "Processed dataset directory "
                f"was not found:\n"
                f"{self.processed_dir}"
            )

        # --------------------------------------------------
        # Find all files for the selected split
        # --------------------------------------------------

        self.files = sorted(
            self.processed_dir.glob(
                f"*_{self.split}.npz"
            )
        )

        if len(self.files) == 0:
            raise FileNotFoundError(
                f"No '*_{self.split}.npz' files "
                f"were found in:\n"
                f"{self.processed_dir}"
            )

        # Number of windows in each subject file
        self.file_lengths = []

        print("=" * 70)
        print(
            f"INITIALIZING {self.split.upper()} IMU DATASET"
        )
        print("=" * 70)

        # --------------------------------------------------
        # Validate each processed subject file
        # --------------------------------------------------

        for file_path in self.files:

            with np.load(
                file_path,
                allow_pickle=False
            ) as data:

                required_keys = {
                    "windows",
                    "activity",
                    "gender",
                    "weight"
                }

                available_keys = set(
                    data.files
                )

                missing_keys = (
                    required_keys
                    - available_keys
                )

                if missing_keys:
                    raise KeyError(
                        f"{file_path.name} is missing "
                        f"the following arrays: "
                        f"{sorted(missing_keys)}"
                    )

                number_of_windows = len(
                    data["windows"]
                )

                number_of_activity_labels = len(
                    data["activity"]
                )

                if (
                    number_of_windows
                    != number_of_activity_labels
                ):
                    raise ValueError(
                        "Window-label mismatch in "
                        f"{file_path.name}: "
                        f"{number_of_windows} windows and "
                        f"{number_of_activity_labels} "
                        "activity labels."
                    )

                if data["windows"].ndim != 3:
                    raise ValueError(
                        f"Expected windows in "
                        f"{file_path.name} to have "
                        "three dimensions."
                    )

                if data["windows"].shape[1:] != (
                    128,
                    30
                ):
                    raise ValueError(
                        f"Unexpected window shape in "
                        f"{file_path.name}: "
                        f"{data['windows'].shape}"
                    )

                self.file_lengths.append(
                    number_of_windows
                )

        # --------------------------------------------------
        # Build cumulative file boundaries
        # --------------------------------------------------

        self.cumulative_lengths = np.cumsum(
            self.file_lengths
        ).tolist()

        self.total_windows = (
            self.cumulative_lengths[-1]
        )

        print(
            "Processed directory :",
            self.processed_dir
        )

        print(
            "Split               :",
            self.split
        )

        print(
            "Subject files       :",
            len(self.files)
        )

        print(
            "Total windows       :",
            self.total_windows
        )

        print(
            "\nDataset initialized successfully."
        )

    def __len__(self):
        """
        Return the total number of windows across all
        subject files.
        """

        return self.total_windows

    def __getitem__(self, index):
        """
        Retrieve one IMU window and its corresponding labels.
        """

        # --------------------------------------------------
        # Support negative indexing
        # --------------------------------------------------

        if index < 0:
            index += self.total_windows

        if (
            index < 0
            or index >= self.total_windows
        ):
            raise IndexError(
                f"Dataset index out of range: {index}"
            )

        # --------------------------------------------------
        # Map global index to subject file and local index
        # --------------------------------------------------

        file_id = bisect.bisect_right(
            self.cumulative_lengths,
            index
        )

        if file_id == 0:
            previous_boundary = 0
        else:
            previous_boundary = (
                self.cumulative_lengths[
                    file_id - 1
                ]
            )

        local_index = (
            index - previous_boundary
        )

        file_path = self.files[file_id]

        # --------------------------------------------------
        # Load the required subject file
        # --------------------------------------------------

        with np.load(
            file_path,
            allow_pickle=False
        ) as data:

            window = np.asarray(
                data["windows"][local_index],
                dtype=np.float32
            ).copy()

            activity = int(
                data["activity"][local_index]
            )

            gender = int(
                data["gender"][0]
            )

            weight = int(
                data["weight"][0]
            )

        # --------------------------------------------------
        # Validate labels
        # --------------------------------------------------

        if activity not in range(6):
            raise ValueError(
                f"Invalid activity label "
                f"{activity} in {file_path.name}"
            )

        if gender not in range(2):
            raise ValueError(
                f"Invalid gender label "
                f"{gender} in {file_path.name}"
            )

        if weight not in range(3):
            raise ValueError(
                f"Invalid weight label "
                f"{weight} in {file_path.name}"
            )

        # --------------------------------------------------
        # Convert arrays and labels to tensors
        # --------------------------------------------------

        # Convert:
        # (128, 30) -> (1, 128, 30)
        window_tensor = torch.from_numpy(
            window
        ).unsqueeze(0)

        activity_tensor = torch.tensor(
            activity,
            dtype=torch.long
        )

        gender_tensor = torch.tensor(
            gender,
            dtype=torch.long
        )

        weight_tensor = torch.tensor(
            weight,
            dtype=torch.long
        )

        return (
            window_tensor,
            activity_tensor,
            gender_tensor,
            weight_tensor
        )


# ==========================================================
# Create Full Multi-Subject Datasets and DataLoaders
# ==========================================================

if RUN_MODE == "full_gpu":

    full_train_dataset = IMUDataset(
        processed_dir=PROCESSED_DIR,
        split="train"
    )

    full_test_dataset = IMUDataset(
        processed_dir=PROCESSED_DIR,
        split="test"
    )

    # Start with two workers on Narval.
    # Increase later only if loading is stable.
    NUM_WORKERS = 2

    full_train_loader = DataLoader(
        full_train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(
            device.type == "cuda"
        ),
        persistent_workers=(
            NUM_WORKERS > 0
        ),
        drop_last=False
    )

    full_test_loader = DataLoader(
        full_test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(
            device.type == "cuda"
        ),
        persistent_workers=(
            NUM_WORKERS > 0
        ),
        drop_last=False
    )

    print("\n" + "=" * 70)
    print("FULL MULTI-SUBJECT DATALOADERS")
    print("=" * 70)

    print(
        "Training subject files :",
        len(full_train_dataset.files)
    )

    print(
        "Testing subject files  :",
        len(full_test_dataset.files)
    )

    print(
        "Training windows       :",
        len(full_train_dataset)
    )

    print(
        "Testing windows        :",
        len(full_test_dataset)
    )

    print(
        "Batch size             :",
        BATCH_SIZE
    )

    print(
        "DataLoader workers     :",
        NUM_WORKERS
    )

    print(
        "Pinned memory          :",
        device.type == "cuda"
    )

    # ------------------------------------------------------
    # Verify one full-data batch
    # ------------------------------------------------------

    (
        sample_windows,
        sample_activities,
        sample_genders,
        sample_weights
    ) = next(
        iter(full_train_loader)
    )

    print("\n" + "=" * 70)
    print("FULL DATASET BATCH VERIFICATION")
    print("=" * 70)

    print(
        "Windows shape   :",
        tuple(sample_windows.shape)
    )

    print(
        "Activity shape  :",
        tuple(sample_activities.shape)
    )

    print(
        "Gender shape    :",
        tuple(sample_genders.shape)
    )

    print(
        "Weight shape    :",
        tuple(sample_weights.shape)
    )

    print(
        "Window dtype    :",
        sample_windows.dtype
    )

    print(
        "Activity dtype  :",
        sample_activities.dtype
    )

    print(
        "Gender dtype    :",
        sample_genders.dtype
    )

    print(
        "Weight dtype    :",
        sample_weights.dtype
    )

    print(
        "\nFull GPU dataset verified successfully."
    )

else:

    print(
        "RUN_MODE is small_cpu. "
        "The balanced small CPU datasets and loaders "
        "will continue to be used."
    )

RUN_MODE is small_cpu. The balanced small CPU datasets and loaders will continue to be used.


In [31]:
# ==========================================================
# Create Full Multi-Subject Datasets
# ==========================================================

if RUN_MODE == "full_gpu":

    surrogate_train_dataset = IMUDataset(
        processed_dir=PROCESSED_DIR,
        split="train"
    )

    surrogate_test_dataset = IMUDataset(
        processed_dir=PROCESSED_DIR,
        split="test"
    )

    print("\nDatasets created successfully.")
    print(
        "Training subject files:",
        len(surrogate_train_dataset.files)
    )
    print(
        "Testing subject files :",
        len(surrogate_test_dataset.files)
    )
    print(
        "Training windows      :",
        len(surrogate_train_dataset)
    )
    print(
        "Testing windows       :",
        len(surrogate_test_dataset)
    )

else:

    surrogate_train_dataset = quick_train_dataset
    surrogate_test_dataset = quick_test_dataset

    print("Using small balanced CPU datasets.")
    print(
        "Training windows:",
        len(surrogate_train_dataset)
    )
    print(
        "Testing windows :",
        len(surrogate_test_dataset)
    )

Using small balanced CPU datasets.
Training windows: 600
Testing windows : 300


In [24]:
# ==========================================================
# Create DataLoaders
# ==========================================================

from torch.utils.data import DataLoader

NUM_WORKERS = 0 if RUN_MODE == "small_cpu" else 2

surrogate_train_loader = DataLoader(
    surrogate_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0)
)

surrogate_test_loader = DataLoader(
    surrogate_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0)
)

print("Training batches :", len(surrogate_train_loader))
print("Testing batches  :", len(surrogate_test_loader))

Training batches : 75
Testing batches  : 38


In [32]:
# ==========================================================
# Verify DataLoader
# ==========================================================

windows, activity, gender, weight = next(
    iter(surrogate_train_loader)
)

print("=" * 65)
print("DATALOADER BATCH VERIFICATION")
print("=" * 65)

print("Windows shape  :", tuple(windows.shape))
print("Activity shape :", tuple(activity.shape))
print("Gender shape   :", tuple(gender.shape))
print("Weight shape   :", tuple(weight.shape))

print("\nWindows dtype  :", windows.dtype)
print("Activity dtype :", activity.dtype)
print("Gender dtype   :", gender.dtype)
print("Weight dtype   :", weight.dtype)


# ----------------------------------------------------------
# Convert labels into class numbers for display
# ----------------------------------------------------------

if activity.ndim > 1:
    first_activity_label = torch.argmax(
        activity[0]
    ).item()
else:
    first_activity_label = activity[0].item()


if gender.ndim > 1:
    first_gender_label = torch.argmax(
        gender[0]
    ).item()
else:
    first_gender_label = gender[0].item()


if weight.ndim > 1:
    first_weight_label = torch.argmax(
        weight[0]
    ).item()
else:
    first_weight_label = weight[0].item()


print("\nFirst activity label :", first_activity_label)
print("First gender label   :", first_gender_label)
print("First weight label   :", first_weight_label)

print("\nDataLoader batch verified successfully.")

DATALOADER BATCH VERIFICATION
Windows shape  : (8, 1, 128, 30)
Activity shape : (8, 6)
Gender shape   : (8, 2)
Weight shape   : (8, 3)

Windows dtype  : torch.float32
Activity dtype : torch.float32
Gender dtype   : torch.float32
Weight dtype   : torch.float32

First activity label : 5
First gender label   : 0
First weight label   : 1

DataLoader batch verified successfully.


In [27]:
print("Train samples :", len(surrogate_train_loader.dataset))
print("Test samples  :", len(surrogate_test_loader.dataset))

Train samples : 600
Test samples  : 300
